#Importando as bibliotecas

In [45]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score

#Selecionando a base de dados de fraude de cartão de crédito

In [34]:
df = pd.read_csv('/content/drive/MyDrive/Modulo 11/creditcard.csv')

df_reduced = df.sample(frac=0.3, random_state=42)

#Tratamento dos dados para diminuir o tempo de processamento do código

In [35]:
X = df_reduced.drop('Class', axis=1)
y = df_reduced['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

#Criando o modelo base

In [36]:
model_base = Sequential()
model_base.add(Dense(16, input_dim=X_train.shape[1], activation='relu'))
model_base.add(Dense(8, activation='relu'))
model_base.add(Dense(1, activation='sigmoid'))

model_base.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model_base.fit(X_train, y_train, epochs=10, batch_size=32, verbose=1)

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
1870/1870 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.9646 - loss: 0.1228
Epoch 2/10
1870/1870 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9995 - loss: 0.0036
Epoch 3/10
1870/1870 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9995 - loss: 0.0026
Epoch 4/10
1870/1870 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9994 - loss: 0.0028
Epoch 5/10
1870/1870 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9994 - loss: 0.0020
Epoch 6/10
1870/1870 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.9994 - loss: 0.0022
Epoch 7/10
1870/1870 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9996 - loss: 0.0016
Epoch 8/10
1870/1870 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.9996 - loss: 0.0017
Epoch 9/10
1870/1870 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.9995 - loss: 0.0020
Epoch 10/10
1870/1870 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9996 - loss: 0.0011


#Iniciando a criação do modelo com hiperparâmetros

In [44]:
def create_model(optimizer='adam', activation='relu', dropout_rate=0.0):
    model = Sequential()
    model.add(Dense(16, input_dim=X_train.shape[1], activation=activation))
    model.add(Dropout(dropout_rate))
    model.add(Dense(8, activation=activation))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

#Definindo os melhores hiperparametros

In [51]:
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score

batch_sizes = [16, 32, 64]
epochs_list = [8, 14]
optimizers = ['adam', 'rmsprop']
dropout_rates = [0.0, 0.2]

best_params = None
best_score = 0

for batch_size in batch_sizes:
    for epochs in epochs_list:
        for optimizer in optimizers:
                for dropout_rate in dropout_rates:
                    model = create_model(optimizer=optimizer, dropout_rate=dropout_rate)

                    model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, verbose=0)

                    y_pred = (model.predict(X_test) > 0.5).astype("int32")

                    accuracy = accuracy_score(y_test, y_pred)
                    recall = recall_score(y_test, y_pred)
                    f1 = f1_score(y_test, y_pred)
                    auc = roc_auc_score(y_test, y_pred)

                    print(f"Batch size: {batch_size}, Epochs: {epochs}, Optimizer: {optimizer}, Dropout: {dropout_rate}")
                    print(f"Accuracy: {accuracy}, Recall: {recall}, F1-score: {f1}, AUC-ROC: {auc}\n")

                    if f1 > best_score:
                        best_score = f1
                        best_params = (batch_size, epochs, optimizer, dropout_rate)

print(f"Melhores parâmetros: {best_params}")

802/802 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Batch size: 16, Epochs: 8, Optimizer: adam, Dropout: 0.0
Accuracy: 0.9994538290484922, Recall: 0.8421052631578947, F1-score: 0.8205128205128205, AUC-ROC: 0.9208963510554076



/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


802/802 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Batch size: 16, Epochs: 8, Optimizer: adam, Dropout: 0.2
Accuracy: 0.9994928412593141, Recall: 0.868421052631579, F1-score: 0.8354430379746836, AUC-ROC: 0.9340542457922497



/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


802/802 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
Batch size: 16, Epochs: 8, Optimizer: rmsprop, Dropout: 0.0
Accuracy: 0.9994928412593141, Recall: 0.8421052631578947, F1-score: 0.8311688311688312, AUC-ROC: 0.9209158861208501



...

802/802 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
Batch size: 64, Epochs: 14, Optimizer: adam, Dropout: 0.2
Accuracy: 0.9994928412593141, Recall: 0.8947368421052632, F1-score: 0.8395061728395061, AUC-ROC: 0.9471926054636495



/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


802/802 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
Batch size: 64, Epochs: 14, Optimizer: rmsprop, Dropout: 0.0
Accuracy: 0.9995318534701362, Recall: 0.8947368421052632, F1-score: 0.8500000000000001, AUC-ROC: 0.9472121405290918



/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


802/802 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Batch size: 64, Epochs: 14, Optimizer: rmsprop, Dropout: 0.2
Accuracy: 0.9995708656809581, Recall: 0.8947368421052632, F1-score: 0.860759493670886, AUC-ROC: 0.9472316755945344

Melhores parâmetros: (32, 8, 'rmsprop', 0.2)


#Utilizando os melhores hiperparametros

In [56]:
best_batch_size = 32
best_epochs = 8
best_optimizer = 'rmsprop'
best_activation = 'relu'
best_dropout_rate = 0.2

def create_model_optimized(optimizer='adam', activation='relu', dropout_rate=0.0):
    model = Sequential()
    model.add(Dense(16, input_dim=X_train.shape[1], activation=activation))
    model.add(Dropout(dropout_rate))
    model.add(Dense(8, activation=activation))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

model_optimized = create_model_optimized(
    optimizer=best_optimizer,
    activation=best_activation,
    dropout_rate=best_dropout_rate
)

model_optimized.fit(
    X_train, y_train,
    epochs=best_epochs,
    batch_size=best_batch_size,
    verbose=1
)


Epoch 1/8


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1870/1870 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.9712 - loss: 0.1259
Epoch 2/8
1870/1870 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9991 - loss: 0.0089
Epoch 3/8
1870/1870 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9991 - loss: 0.0076
Epoch 4/8
1870/1870 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9992 - loss: 0.0074
Epoch 5/8
1870/1870 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9992 - loss: 0.0073
Epoch 6/8
1870/1870 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9994 - loss: 0.0075
Epoch 7/8
1870/1870 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9995 - loss: 0.0061
Epoch 8/8
1870/1870 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9992 - loss: 0.0073


#Comparação do modelo com hiperparâmetros e o modelo base

##Resultado do modelo com ajuste utilizando a hiperparametrização

In [57]:
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score

y_pred_optimized = (model_optimized.predict(X_test) > 0.5).astype("int32")

accuracy_opt = accuracy_score(y_test, y_pred_optimized)
recall_opt = recall_score(y_test, y_pred_optimized)
f1_opt = f1_score(y_test, y_pred_optimized)
auc_opt = roc_auc_score(y_test, y_pred_optimized)

print(f"Modelo otimizado - Accuracy: {accuracy_opt}")
print(f"Modelo otimizado - Recall: {recall_opt}")
print(f"Modelo otimizado - F1-score: {f1_opt}")
print(f"Modelo otimizado - AUC-ROC: {auc_opt}")


802/802 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Modelo otimizado - Accuracy: 0.9994538290484922
Modelo otimizado - Recall: 0.868421052631579
Modelo otimizado - F1-score: 0.825
Modelo otimizado - AUC-ROC: 0.9340347107268073


##Resultado do modelo base

In [37]:
y_pred_base = (model_base.predict(X_test) > 0.5).astype("int32")

print("Modelo Base:")
print(classification_report(y_test, y_pred_base))
print(f"AUC-ROC: {roc_auc_score(y_test, y_pred_base)}")


802/802 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
Modelo Base:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     25595
           1       0.78      0.92      0.84        38

    accuracy                           1.00     25633
   macro avg       0.89      0.96      0.92     25633
weighted avg       1.00      1.00      1.00     25633

AUC-ROC: 0.9603309651350489


#Resultados

##Na comparação entre os modelos, é possível notar que o modelo base possúi métricas melhores, no entanto isso não quer dizer que o modelo é melhor, uma vez que a utilização de hiperparametros busca tornar os resultados mais confiáveis.

##Os resultados extremamente altos de desempenho observados são, em grande parte, atribuídos à redução do tamanho do conjunto de dados. O conjunto de dados reduzido pode levar a um sobreajuste, onde o modelo aprende bem as particularidades dos dados de treinamento, mas não necessariamente generaliza bem para novos dados.

##No cenário real, um modelo treinado com um conjunto de dados reduzido pode apresentar métricas de desempenho muito boas, mas essas métricas podem não refletir a performance do modelo em um cenário de dados mais amplo e realista. É essencial validar os modelos com conjuntos de dados maiores e mais diversos para garantir que eles se comportem bem em condições reais e não apenas no conjunto de dados reduzido usado para treinamento e avaliação.